# VEGFR2 Activity Prediction - Colab Notebook

This notebook provides a complete workflow for:
1. **Environment setup** - Install dependencies
2. **Data download** - Fetch ChEMBL VEGFR2 IC50 data
3. **Model training** - Train ML (RF, SVM, XGBoost) and GNN (GCN, GAT, MPNN) models
4. **Enriched GNN** - GNN with Morgan+MACCS injected into graph nodes (NEW!)
5. **Screening/Inference** - Predict on new compound libraries

## Key Innovation: Enriched Graph Approach
Instead of using only atom features (32-dim), we inject Morgan (2048) + MACCS (166) 
fingerprints into each atom node. This gives the GNN access to fingerprint knowledge 
while still learning graph-based patterns through message passing.

## Requirements
- GPU runtime (required for GNN training)
- ~10-15 min for full training

**Enable GPU**: Runtime → Change runtime type → GPU

## 1. Environment Setup

In [5]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. GNN training will be VERY slow on CPU.")
    print("   Enable GPU: Runtime → Change runtime type → GPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [6]:
# Install dependencies
%pip install -q numpy pandas pyyaml scikit-learn xgboost rdkit optuna
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 
%pip install torch_geometric -q 
# Verify imports
import torch, numpy, pandas, sklearn, xgboost, rdkit, yaml, optuna
print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 29.4 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.9 MB/s eta 0:00:00a 0:00:01
✅ All packages installed


In [7]:
# Clone the repository and install package
!rm -f /content/ai-code/data/processed/*.csv
!cd /content/ai-code && git pull
!cd /content/ai-code && pip install -e .
!git clone https://github.com/Techbjd/ai-code.git /content/ai-code 2>/dev/null; cd /content/ai-code && git pull && pip install -e .

# Ensure Python can find the package
import sys
sys.path.insert(0, "/content/ai-code/src")
import os
os.chdir("/content/ai-code")
print(f"Working in: {os.getcwd()}")

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 6.17 KiB | 2.06 MiB/s, done.
From https://github.com/Techbjd/ai-code
   8b5a5a6..1303b1b  main       -> origin/main
Updating 8b5a5a6..1303b1b
Fast-forward
 src/vegfr2/gnn_pyg.py    |   43 +-
 vegfr2_colab_fixed.ipynb | 1858 ++++++++++++++++++++++++----------------------
 2 files changed, 1027 insertions(+), 874 deletions(-)
Obtaining file:///content/ai-code
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vegfr2 (pyproject.toml) ... done
  Created wheel for vegfr2: filename=vegfr2-0.1.0-0.editable-py3-none-any.whl size=4479 sha256=51488d447e7f683b3cfc6d0de6c0c95a29259840fd1ca1a

In [8]:
# Verify package import works
from vegfr2.features import mol_to_graph, ATOM_FEAT_DIM, BOND_FEAT_DIM
from vegfr2.gnn_models import build_model
from vegfr2.ml_models import train_ml_model
print(f"✅ Package imported successfully")
print(f"   Atom feat dim: {ATOM_FEAT_DIM}")
print(f"   Bond feat dim: {BOND_FEAT_DIM}")

✅ Package imported successfully
   Atom feat dim: 32
   Bond feat dim: 11


## 2. Download Training Data

In [9]:
# Download pre-processed train/val/test splits from GitHub
!mkdir -p data/processed
!wget -q -O data/processed/train.csv https://raw.githubusercontent.com/Techbjd/ai-code/main/data/processed/train.csv
!wget -q -O data/processed/val.csv https://raw.githubusercontent.com/Techbjd/ai-code/main/data/processed/val.csv
!wget -q -O data/processed/test.csv https://raw.githubusercontent.com/Techbjd/ai-code/main/data/processed/test.csv

# Verify data
import pandas as pd
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")
print(f"Train: {len(train_df)} (active: {train_df['active'].sum()}, inactive: {(train_df['active']==0).sum()})")
print(f"Val: {len(val_df)} (active: {val_df['active'].sum()}, inactive: {(val_df['active']==0).sum()})")
print(f"Test: {len(test_df)} (active: {test_df['active'].sum()}, inactive: {(test_df['active']==0).sum()})")

Train: 7834 (active: 4441, inactive: 3393)
Val: 980 (active: 556, inactive: 424)
Test: 980 (active: 556, inactive: 424)


## 3. Train Models

Choose which models to train. Each cell trains one model type.

**Note**: HPO (hyperparameter optimization) is recommended for GNNs to match paper results.

In [ ]:
# Train Classical ML models (RF, SVM, XGBoost) using pre-processed splits
# Fast - runs on CPU, ~1-2 minutes total
!python scripts/train.py --model rf --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model svm --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model xgb --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv

# Results saved to runs/<model>/model.pkl
# Metrics saved to runs/results.json

In [ ]:
# Train GNN models WITH HPO (recommended for best results)
# ~10-15 minutes per model on GPU
!cd /content/ai-code && git pull
!python scripts/train.py --model gcn --hpo --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model gat --hpo --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
!python scripts/train.py --model mpnn --hpo --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
# Results saved to runs/<model>/best.pt
# Metrics saved to runs/results.json

In [ ]:
# OR train GNN models WITHOUT HPO (faster, uses config defaults)
# ~5-10 minutes per model on GPU
# !python scripts/train.py --model gcn --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
# !python scripts/train.py --model gat --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv
# !python scripts/train.py --model mpnn --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv

In [ ]:
# OR train all models at once using pre-processed splits
!python scripts/train.py --model all --config configs/config.yaml --train-csv data/processed/train.csv --val-csv data/processed/val.csv --test-csv data/processed/test.csv

# View results summary
import json
with open("runs/results.json") as f:
    results = json.load(f)

print(f"{'Model':<8} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}")
print("-" * 44)
for name, m in results.items():
    auc_str = f"{m['auc']:.4f}" if m["auc"] is not None else "N/A"
    print(f"{name:<8} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

## 4. Screen New Compound Libraries

Use trained models to predict VEGFR2 activity on new SMILES.

In [ ]:
# Prepare a screening library (example: create test CSV)
import pandas as pd

# Example: Your compound library
library_smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",  # Aspirin
    "CCO",                       # Ethanol
    "C1=CC=CC=C1",               # Benzene
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",  # Ibuprofen
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",   # Caffeine
    "CC(C)CC1=CC(=CC=C1)O",    # Naproxen-like
    "C[C@H](O)CC1=CC=CC=C1",   # Chiral molecule (R)
    "C[C@@H](O)CC1=CC=CC=C1",  # Chiral molecule (S)
    "C/C=C/C",                 # E-alkene
    "C/C=C\\C",                # Z-alkene
]

library_df = pd.DataFrame({"smiles": library_smiles})
library_df.to_csv("data/screen_library.csv", index=False)
print(f"Created library with {len(library_df)} compounds")
print(library_df)

In [ ]:
# Screen with a trained GNN model (FIXED - works now)
# Change model_path to your best model
MODEL_PATH = "runs/gcn/best.pt"  # or gat/best.pt, mpnn/best.pt
INPUT_CSV = "data/screen_library.csv"
OUTPUT_CSV = "results/screen_gcn_results.csv"
THRESHOLD = 0.5

!mkdir -p results
!python scripts/screen.py \
    --model {MODEL_PATH} \
    --input {INPUT_CSV} \
    --output {OUTPUT_CSV} \
    --threshold {THRESHOLD} \
    --batch-size 32

# View results
results_df = pd.read_csv(OUTPUT_CSV)
print(results_df[['smiles', 'probability', 'hit']].to_string(index=False))

In [ ]:
# Screen with a trained ML model
MODEL_PATH = "runs/xgb/model.pkl"  # or rf/model.pkl, svm/model.pkl
INPUT_CSV = "data/screen_library.csv"
OUTPUT_CSV = "results/screen_xgb_results.csv"
THRESHOLD = 0.5

!mkdir -p results
!python scripts/screen.py \
    --model {MODEL_PATH} \
    --input {INPUT_CSV} \
    --output {OUTPUT_CSV} \
    --threshold {THRESHOLD} \
    --batch-size 256

# View results
results_df = pd.read_csv(OUTPUT_CSV)
print(results_df[['smiles', 'probability', 'hit']].to_string(index=False))

In [ ]:
# Screen with multiple models and compare (ensemble)
import pandas as pd
import numpy as np

models_to_screen = [
    ("runs/gcn/best.pt", "GCN"),
    ("runs/gat/best.pt", "GAT"),
    ("runs/mpnn/best.pt", "MPNN"),
    ("runs/xgb/model.pkl", "XGBoost"),
    ("runs/rf/model.pkl", "RandomForest"),
]http://221.217.49.129:9000

library_df = pd.read_csv("data/screen_library.csv")
all_results = library_df[['smiles']].copy()

for model_path, model_name in models_to_screen:
    try:
        if model_path.endswith('.pt'):
            !python scripts/screen.py --model {model_path} --input data/screen_library.csv --output results/temp_{model_name}.csv --threshold 0.5 --batch-size 32
        else:
            !python scripts/screen.py --model {model_path} --input data/screen_library.csv --output results/temp_{model_name}.csv --threshold 0.5 --batch-size 256

        res = pd.read_csv(f"results/temp_{model_name}.csv")
        all_results[f"prob_{model_name}"] = res["probability"]
        all_results[f"hit_{model_name}"] = res["hit"]
        print(f"✅ {model_name} done")
    except Exception as e:
        print(f"❌ {model_name} failed: {e}")

# Ensemble: average probabilities across models
ensemble_cols = [c for c in all_results.columns if c.startswith('prob_')]
all_results['prob_ensemble'] = all_results[ensemble_cols].mean(axis=1)
all_results['hit_ensemble'] = all_results['prob_ensemble'] >= 0.5

# Sort by ensemble probability
all_results = all_results.sort_values('prob_ensemble', ascending=False)

Path("results").mkdir(exist_ok=True)
all_results.to_csv("results/combined_screening.csv", index=False)

display_cols = ['smiles', 'prob_ensemble', 'hit_ensemble'] + ensemble_cols
print("\nEnsemble results:")
print(all_results[display_cols].to_string(index=False))

## 5. Advanced: Custom Training & Programmatic Usage

In [ ]:
# Custom training with your own config
import yaml

custom_config = {
    "seed": 42,
    "paths": {
        "raw_csv": "data/raw/chembl_vegfr2.csv",
        "output_dir": "runs_custom"
    },
    "label": {"threshold_nM": 500},
    "split": {"test_size": 0.1, "val_frac_of_remaining": 0.111111},
    "fingerprint": {"radius": 2, "n_bits": 2048},
    "gnn": {
        "hidden": 128,
        "layers": 4,
        "heads": 8,
        "batch": 64,
        "lr": 0.0005,
        "epochs": 300,
        "patience": 20
    },
    "hpo": {"n_trials": 30}
}

with open("configs/custom_config.yaml", "w") as f:
    yaml.dump(custom_config, f)

print("Custom config saved. Train with:")
print("!python scripts/train.py --model gcn --config configs/custom_config.yaml")

In [ ]:
# Programmatic usage: train and predict without CLI
from vegfr2.gnn_models import train_gnn_model, predict_gnn_model, save_checkpoint, load_checkpoint
from vegfr2.ml_models import train_ml_model, predict_ml_model, save_ml_model, load_ml_model
from vegfr2.features import smiles_to_morgan, mol_to_graph
import numpy as np
import torch

# Load data
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")

# --- Train ML model (Random Forest) ---
X_train = np.vstack([smiles_to_morgan(s) for s in train_df['smiles']])
y_train = train_df['active'].values
X_val = np.vstack([smiles_to_morgan(s) for s in val_df['smiles']])
y_val = val_df['active'].values

rf = train_ml_model('rf', X_train, y_train, seed=42)
val_probs = predict_ml_model(rf, X_val)
from vegfr2.metrics import classification_metrics
print("RF Val metrics:", classification_metrics(y_val.tolist(), val_probs.tolist()))
save_ml_model(rf, "runs_custom/rf_model.pkl")

# --- Train GNN model (GCN) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
gcn = train_gnn_model(
    name="gcn",
    train_smiles=train_df['smiles'].tolist(),
    train_labels=train_df['active'].astype(int).tolist(),
    val_smiles=val_df['smiles'].tolist(),
    val_labels=val_df['active'].astype(int).tolist(),
    hidden=64,
    layers=3,
    lr=0.001,
    batch_size=128,
    epochs=200,
    patience=15,
    device=device
)
save_checkpoint(gcn, "runs_custom/gcn_model.pt")

# --- Predict on new SMILES ---
test_smiles = ["CCO", "CC(=O)OC1=CC=CC=C1C(=O)O", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]

# ML prediction
X_test = np.vstack([smiles_to_morgan(s) for s in test_smiles])
ml_probs = predict_ml_model(rf, X_test)
print("\nML predictions:", dict(zip(test_smiles, ml_probs)))

# GNN prediction
gnn_probs = predict_gnn_model(gcn, test_smiles, device=device)
print("GNN predictions:", dict(zip(test_smiles, gnn_probs)))

In [ ]:
# Optional: Use PyTorch Geometric implementations instead
# Uncomment and run if you want to try PyG models

# %pip install torch_geometric -q
# from vegfr2.gnn_pyg import train_gnn_pyg, predict_gnn_pyg, save_checkpoint, load_checkpoint
# 
# model = train_gnn_pyg(
#     name="gcn",
#     train_smiles=train_df['smiles'].tolist(),
#     train_labels=train_df['active'].astype(int).tolist(),
#     val_smiles=val_df['smiles'].tolist(),
#     val_labels=val_df['active'].astype(int).tolist(),
#     device="cuda"
# )
# print("PyG model trained!")

In [ ]:
# ============================================================
# NEW: Enriched GNN with Morgan+MACCS in Graph Nodes
# ============================================================
# This injects fingerprint knowledge INTO the graph so GNN
# can learn graph patterns WHILE knowing fingerprint info.

import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from vegfr2.features import mol_to_graph_with_fps, get_enriched_node_dim
from vegfr2.gnn_pyg import EnrichedPyGDataset, build_pyg_model
from vegfr2.metrics import classification_metrics

# Load data
import pandas as pd
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Get enriched feature dimension
ENRICHED_DIM = get_enriched_node_dim(use_morgan=True, use_maccs=True)
print(f"Enriched node dimension: {ENRICHED_DIM} (32 atom + 2048 morgan + 166 maccs)")

# Create enriched datasets
train_ds = EnrichedPyGDataset(
    train_df["smiles"].tolist(),
    train_df["active"].astype(int).tolist(),
    use_morgan=True,
    use_maccs=True,
)
val_ds = EnrichedPyGDataset(
    val_df["smiles"].tolist(),
    val_df["active"].astype(int).tolist(),
    use_morgan=True,
    use_maccs=True,
)
test_ds = EnrichedPyGDataset(
    test_df["smiles"].tolist(),
    test_df["active"].astype(int).tolist(),
    use_morgan=True,
    use_maccs=True,
)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Datasets created with enriched features!")
print(f"Sample node shape: {train_ds[0].x.shape}")  # Should be (n_atoms, 2246)

# Build model with enriched input dimension
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_pyg_model(
    "gcn",
    in_dim=ENRICHED_DIM,  # 2246 instead of 32!
    hidden=128,
    layers=3,
    dropout=0.3,
).to(device)

print(f"Model built with in_dim={ENRICHED_DIM}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
loss_fn = nn.BCEWithLogitsLoss()

best_auc = 0.0
patience = 15
wait = 0
EPOCHS = 100

print("\nTraining Enriched GCN...")
for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(logits.squeeze(), batch.y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    
    # Validate
    model.eval()
    val_probs = []
    val_true = []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index, batch.batch)
            val_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
            val_true.extend(batch.y.cpu().numpy())
    
    val_metrics = classification_metrics(val_true, val_probs)
    val_auc = val_metrics.get("auc") or 0.0
    scheduler.step()
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "runs/enriched_gcn/best.pt")
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_df):.4f} | Val AUC: {val_auc:.4f}")

# Test evaluation
model.load_state_dict(torch.load("runs/enriched_gcn/best.pt"))
model.eval()
test_probs = []
test_true = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        test_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
        test_true.extend(batch.y.cpu().numpy())

test_metrics = classification_metrics(test_true, test_probs)
print("\n" + "="*50)
print("ENRICHED GCN RESULTS:")
print("="*50)
print(f"ACC:  {test_metrics['acc']:.4f}")
print(f"SEN:  {test_metrics['sen']:.4f}")
print(f"SPE:  {test_metrics['spe']:.4f}")
print(f"MCC:  {test_metrics['mcc']:.4f}")
print(f"AUC:  {test_metrics['auc']:.4f}")
print("="*50)


## Summary

### Expected Results (approximate)
| Model | AUC | Notes |
|-------|-----|-------|
| RF | ~0.91 | Best classical ML |
| SVM | ~0.90 | Good classical ML |
| XGBoost | ~0.89 | Good classical ML |
| GCN | ~0.70-0.73 | With HPO |
| GAT | ~0.56-0.61 | Struggles on this dataset |
| MPNN | ~0.74-0.76 | Best GNN |

### Key Fixes Applied
1. **Fixed `AttributeError: 'int' object has no attribute 'to'`** in both `train.py` and `screen.py` - now only moves tensors to device
2. **Added programmatic APIs** (`train_gnn_model`, `predict_gnn_model`) for ML-style usage
3. **Created PyG alternative** (`gnn_pyg.py`) for users preferring PyTorch Geometric
4. **HPO recommended** for GNNs to get competitive results

### Next Steps
- Try ensemble predictions (Section 4)
- Customize hyperparameters (Section 5)
- Use PyG implementations if preferred
- Screen your own compound libraries